# Lesson 4 — Files, libraries and the scientific stack

**Python for Gravitational Waves · Lesson 4 of 17**


## What this lesson is for

Everything so far lived inside the notebook. Real work does not: data arrives
in files and the code that handles it is mostly written by other people.

So this lesson is about the two things that connect your code to the world.
**Files** — text, CSV, JSON and the HDF5 format that gravitational wave data
actually comes in. And **libraries** — how to import them, where they come
from and the four that carry this course.

It ends by opening the real strain data from 14 September 2015 and plotting
it. Not analysing it: getting a signal out of that file needs a model of the
noise and a model of the signal, which are built in Lessons 10 to 13. Opening
it, understanding its structure and looking at it is quite enough for one
lesson, and it is the foundation the rest of the course stands on.


## Text files

The oldest and simplest case. Open a file, read it, close it.


In [ ]:
# make a small file to work with
with open('notes.txt', 'w') as f:
    f.write('H1 observing\n')
    f.write('L1 observing\n')
    f.write('V1 down for maintenance\n')

print('written')


The `with` block is the important part. It opens the file, gives it to you as
`f` and closes it when the block ends — **even if something inside fails**.
Files opened without `with` stay open when an error interrupts you and on a
shared machine that eventually becomes somebody else's problem.

The mode is the second argument:

| Mode | Meaning |
|---|---|
| `'r'` | read, the default |
| `'w'` | write, **and it deletes what was there** |
| `'a'` | append, add to the end |

Three ways to read it back:


In [ ]:
with open('notes.txt', 'r') as f:
    whole = f.read()               # everything, as one string
print(repr(whole))

with open('notes.txt', 'r') as f:
    lines = f.readlines()          # a list of lines
print(lines)

with open('notes.txt', 'r') as f:
    for line in f:                 # one line at a time -- best for big files
        print(line.strip())


`line.strip()` removes the newline character at the end. Forgetting it is the
reason your printed output has blank lines in it.

The third form reads one line at a time and never holds the whole file in
memory. For a log file of a few kilobytes it makes no difference; for a data
file of several gigabytes it is the difference between working and not.


### When the file is not there


In [ ]:
try:
    with open('does_not_exist.txt', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print("not there -- check the path and where you are")

import os
print("we are in:", os.getcwd())


Most "the file is not there" errors are really "I am not where I think I am".
`os.getcwd()` tells you the working directory and `os.listdir()` tells you what
is in it. Check those two before doubting the file.


## CSV: rows and columns

**CSV** — comma separated values — is a table saved as text. The course ships
one: every confident detection to date.


In [ ]:
path = os.path.join('..', 'labs', 'data', 'gwtc_events.csv')

with open(path, 'r') as f:
    for i, line in enumerate(f):
        print(line.strip()[:96])
        if i == 3:
            break


The first line is the **header**, naming the columns. The `csv` module in the
standard library reads it properly, handling quoted fields and commas inside
values, which naive splitting on commas does not:


In [ ]:
import csv

with open(path, 'r') as f:
    reader = csv.DictReader(f)          # each row becomes a dictionary
    rows = list(reader)

print(len(rows), "events")
first = rows[0]
print(first['commonName'], first['mass_1_source'],
      first['luminosity_distance'])


`DictReader` gives each row as a dictionary keyed by the column names, which
is far easier to read than `row[7]`.

Note that everything is a **string**. CSV has no types; converting is your
job:


In [ ]:
m1 = float(rows[0]['mass_1_source'])
print(m1, type(m1))


### The same file with pandas

For tables, the `pandas` library does the whole job in one line and gives you
something you can compute with.


In [ ]:
import pandas as pd

events = pd.read_csv(path)
print(events.shape)                     # rows, columns
print(events['mass_1_source'].head())
events.head(3)


In [ ]:
# a column is an array you can do arithmetic on
heavy = events[events['mass_1_source'] > 50]
print(f"{len(heavy)} events with a primary above 50 solar masses")
print(f"heaviest primary: {events['mass_1_source'].max():.1f}")


`events['column'] > 50` is the boolean mask of Lesson 3 and
`events[mask]` keeps the rows where it is `True`. That is nearly all the pandas
this course needs: read a table, take a column, select rows.


## JSON: nested data

**JSON** stores structured data — dictionaries, lists, numbers, strings,
nested inside each other. Catalogue queries and configuration files use it.


In [ ]:
import json

event = {
    "name": "GW150914",
    "gps": 1126259462.4,
    "masses": [36.0, 29.0],
    "detectors": {"H1": {"snr": 19.4}, "L1": {"snr": 13.3}},
}


Write it out, then read it back and dig into the nesting.


In [ ]:
with open('event.json', 'w') as f:
    json.dump(event, f, indent=2)

with open('event.json', 'r') as f:
    back = json.load(f)

print(back["name"], back["masses"])
print(back["detectors"]["H1"]["snr"])


JSON maps onto Python containers exactly: an object becomes a dictionary, an
array becomes a list. That is why it is everywhere.


## Modules and libraries

A **module** is a Python file you can import. A **library** is a collection of
them, written by somebody else and installed on your machine.


In [ ]:
import math
print(math.sqrt(16), math.pi)

from math import sqrt, pi          # bring specific names in
print(sqrt(16), pi)

import numpy as np                 # give it a shorter name
print(np.sqrt(16))


Three forms, all used. `import x` keeps the name in front, which is clearest.
`from x import y` is convenient for a name used constantly. `import x as y` is
for the conventional short names — `np`, `pd`, `plt` — which everybody writes
the same way and you should too.

Avoid `from x import *`: it dumps every name into your notebook and you can no
longer tell where anything came from.

**Where libraries come from.** They are installed with `pip`, from a public
index of hundreds of thousands of packages:

```bash
pip install h5py                # install
pip list                        # what is installed
```

You will not need to do this often here: the environment for the course
already has what the labs use. What matters is knowing that `import h5py`
works because somebody installed `h5py` and that an `ImportError` means it is
missing rather than broken.

The four that carry this course and what each is for:

| Library | What it does |
|---|---|
| `numpy` | arrays and the arithmetic on them |
| `matplotlib` | plots |
| `scipy` | signal processing, integration, optimisation, statistics |
| `pandas` | tables |

Later come `h5py` for HDF5 files, `gwpy` for gravitational wave time series
and `bilby` for parameter estimation.


## HDF5: how strain data is stored

A text file with a hundred and thirty thousand numbers in it would be slow and
enormous. Scientific data is stored in binary and the standard format here is
**HDF5**.

An HDF5 file is a small file system inside one file:

- **groups** are like folders,
- **datasets** are arrays,
- **attributes** are small pieces of metadata attached to either.

Make one, to see the structure from the inside:


In [ ]:
import h5py

with h5py.File('example.h5', 'w') as f:
    g = f.create_group('experiment')
    d = g.create_dataset('samples', data=np.linspace(0, 1, 100))
    d.attrs['units'] = 'seconds'
    d.attrs['sample_rate'] = 100.0

print('written')


Now open it again and look at what is inside.


In [ ]:
with h5py.File('example.h5', 'r') as f:
    print(list(f.keys()))
    print(list(f['experiment'].keys()))
    data = f['experiment/samples'][:]
    print(data.shape, dict(f['experiment/samples'].attrs))


> **Important — The colon that reads the data**
>
> `f['experiment/samples']` gives you a *reference* to the dataset, not the
> numbers. `f['experiment/samples'][:]` reads them into memory as an array.
>
> This is deliberate and it is why HDF5 handles files larger than your memory:
> you can look at the shape and the attributes of a hundred gigabyte dataset
> and then read only the slice you want.


### The real file

Now the data LIGO released for 14 September 2015, committed with the course.


In [ ]:
strain_path = os.path.join('..', 'labs', 'data', 'gw150914_H1.hdf5')

def show(name, obj):
    print(f"{name:34} {getattr(obj, 'shape', '')}")

with h5py.File(strain_path, 'r') as f:
    f.visititems(show)


`visititems` walks the whole tree. You can see the strain itself, the metadata
and the data quality flags.

Read the array and the two pieces of metadata that give it a time axis:


In [ ]:
with h5py.File(strain_path, 'r') as f:
    strain = f['strain/Strain'][:]
    dt = f['strain/Strain'].attrs['Xspacing']   # seconds per sample
    gps0 = f['meta/GPSstart'][()]               # [()] reads a scalar
    detector = f['meta/Detector'][()].decode()  # bytes -> str

fs = 1.0 / dt
t = gps0 + np.arange(strain.size) * dt


Everything the array needs came out of the file. Now say what we have.


In [ ]:
print(f"detector      {detector}")
print(f"samples       {strain.size}")
print(f"sample rate   {fs:.0f} Hz")
print(f"duration      {strain.size / fs:.0f} s")
print(f"starts at GPS {gps0}")


Everything here is a habit worth copying. The sample rate came from the file
rather than from memory. The time axis was built from the start time and the
spacing rather than assumed. The detector name was read rather than guessed.

Data you did not write always deserves that suspicion.


## Plotting it

`matplotlib` draws it. The pattern is always the same: make a figure and axes,
draw on the axes, label them.


In [ ]:
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, os.path.abspath(os.path.join('..')))
from gwcourse_palette import P, apply_style
apply_style()

event_gps = 1126259462.4


Make the figure and the axes, draw, label. Always in that order.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t - event_gps, strain, lw=0.6, color=P['BLUE'])
ax.set_xlabel('time from the event (s)')
ax.set_ylabel('strain')
ax.set_title(f'{detector}: 32 seconds around GW150914')
plt.show()


That is the data in which the first gravitational wave was found and there is
nothing to see. The plot is dominated by low frequency noise thousands of times
larger than any signal.

> **Careful — This is the normal situation and it is the point of the course**
>
> The signal is not visible and was never going to be. Getting it out needs a
> model of the noise and a model of the signal, and those come in Lessons 10 to
> 13 — the power spectral density, whitening, filtering, matched filtering.
>
> Do not try those techniques here. Lesson 11 teaches them properly, once you
> know what a spectrum is. What this lesson gives you is the thing that comes
> first: the file opened, the array in hand, the axes labelled.

Two panels, which is how you will usually compare things:


In [ ]:
l1_path = os.path.join('..', 'labs', 'data', 'gw150914_L1.hdf5')
with h5py.File(l1_path, 'r') as f:
    strain_l1 = f['strain/Strain'][:]

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, s, name, colour in ((axes[0], strain, 'H1', P['BLUE']),
                            (axes[1], strain_l1, 'L1', P['SAGE_DARK'])):
    ax.plot(t - event_gps, s, lw=0.6, color=colour)
    ax.set_ylabel('strain')
    ax.set_title(name, loc='left', fontsize=11)
axes[1].set_xlabel('time from the event (s)')
fig.tight_layout()
plt.show()


### Describing data before analysing it


In [ ]:
from scipy import stats

print(f"mean      {strain.mean():.3e}")
print(f"std       {strain.std():.3e}")
print(f"min, max  {strain.min():.3e}, {strain.max():.3e}")
print(f"skewness  {stats.skew(strain):.3f}")

# and where the largest excursion is
i = np.abs(strain).argmax()
print(f"largest |strain| at t = {t[i] - event_gps:+.3f} s from the event")


The largest excursion in the raw data is nowhere near the event, which is the
same lesson again: whatever is loudest is not the signal.

`scipy` is the other half of the scientific stack — statistics here,
integration in Lesson 5, signal processing from Lesson 6 onwards.


## The library that knows what strain is

Everything above treats the file as a table of numbers, which it is. There is
also a library that knows the numbers are a **time series from a
gravitational wave detector** and once you are doing physics rather than
file handling it saves a great deal of typing.

`gwpy` reads the same file and hands back an object that carries its sample
rate and its start time, so the operations of Lessons 11 and 12 are one call
each. It is worth seeing plainly once, before you build them by hand.

In [ ]:
from gwpy.timeseries import TimeSeries

# the same cached file, read by a library that knows the format
strain = TimeSeries.read('../demo/data/gw150914_H1.hdf5', format='hdf5.gwosc')

print(strain)
print()
print(f"duration    {strain.duration}")
print(f"sample rate {strain.sample_rate}")
print(f"starts at   GPS {strain.t0}")

The object prints itself with its units and its epoch and every method below
knows them. Compare that with keeping the sample rate in a separate variable
and remembering to divide by it.

Four operations, one line each — the amplitude spectral density, the
whitening, the band-pass and the time–frequency map. Each is a piece of
Lessons 11 and 12 compressed into a method call, which is convenient once you
know what the method does and dangerous before.

In [ ]:
asd = strain.asd(fftlength=4)                       # Welch, as in H06
white = strain.whiten(fftlength=4).bandpass(35, 350)
q_map = strain.q_transform(outseg=(1126259462.2, 1126259462.5),
                           frange=(20, 400))          # as in H07

print(f"ASD at 100 Hz: {asd.value[abs(asd.frequencies.value - 100).argmin()]:.2e}"
      f" {asd.unit}")
print(f"whitened data: standard deviation {white.value.std():.2f}")
print(f"time-frequency map: {q_map.shape[0]} times by {q_map.shape[1]} frequencies")

> **Careful — Convenience is not understanding**
>
> `whiten` chooses a segment length, a window, a detrending and how to handle
> the edges. Every one of those choices changes the answer, and in Lesson 11 you
> will make them yourself and watch them do it.
>
> Use the library and know what it is doing on your behalf. The order in this
> course is deliberate: you build the operation out of `numpy` first, and only
> then does the one-line version become a saving rather than a black box.

## A note on frame files

The other format you will meet is the **GWF frame file**, the format the
detectors themselves write. It stores fixed length blocks of time series with
their metadata and it is read with `gwpy` rather than `h5py`:

```python
from gwpy.timeseries import TimeSeries
data = TimeSeries.read('file.gwf', channel='H1:GWOSC-4KHZ_R1_STRAIN')
```

We use HDF5 in this course because the public data releases are distributed in
it and because `h5py` shows the structure plainly. If you go on to work inside
a collaboration you will meet GWF constantly.


## Exercises


### 1. Write, read, check

Write the twenty numbers $0, 5, 10, \dots, 95$ to a text file, one per line.
Read them back, convert them to numbers and check the sum is what you expect.


In [ ]:
# your code here


**Solution.** Write your own version first — the error you make on the way is
the part you keep.


### 2. The catalogue, three ways

Using `gwtc_events.csv`: how many events have both masses below 3 solar masses?
Do it once with `csv.DictReader` and once with `pandas` and check the two
agree.

**What to look for.** Those are the candidates involving neutron stars rather
than black holes.


In [ ]:
# your code here


**Solution.** Write your own version first — the error you make on the way is
the part you keep.


### 3. Read the metadata properly

Open `gw150914_L1.hdf5` and print the detector name, the start time, the
duration and the sample rate, taking every one of them from the file rather
than assuming. Then check that its strain array is the same length as H1's.


In [ ]:
# your code here


**Solution.** Write your own version first — the error you make on the way is
the part you keep.


### 4. A small HDF5 file of your own

Create an HDF5 file with a group called `run1` containing two datasets — a
time array and a sine wave — with the sample rate stored as an attribute. Read
it back in a separate cell and plot it using only what you read from the file.


In [ ]:
# your code here


**Solution.** Write your own version first — the error you make on the way is
the part you keep.


## Where this leaves you

You can open a file in four formats, read what is in it and plot it with the
axes labelled. That is the floor every lesson from here on builds on, and it is
where the Python primer ends.

**Do.** Open `gw150914_H1.hdf5` and `gw150914_L1.hdf5` in the same cell, build
each time axis from that file's own metadata and check that the two records
cover the same thirty-two seconds. Any analysis that uses two detectors begins
with exactly that check.

**Next.** The physics. Lesson 5 integrates the equation of a suspended mirror
with `scipy` and reads its response off the solution. Lessons 11 and 12 take
the file you have just opened and do the signal processing this lesson deliberately avoided: the
spectrum, whitening, and the first sight of the signal itself.
